In [7]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 1500

data = pd.DataFrame({
    "customer_id": range(1, n+1),
    "gender": np.random.choice(["Male", "Female"], n),
    "senior_citizen": np.random.choice([0, 1], n, p=[0.85, 0.15]),
    "tenure_months": np.random.randint(1, 72, n),
    "monthly_charges": np.random.uniform(20, 120, n).round(2),
    "contract_type": np.random.choice(
        ["Month-to-month", "One year", "Two year"], 
        n, 
        p=[0.6, 0.25, 0.15]
    ),
    "internet_service": np.random.choice(
        ["DSL", "Fiber optic", "No"], 
        n, 
        p=[0.4, 0.5, 0.1]
    ),
    "online_security": np.random.choice([0,1], n, p=[0.7,0.3]),
    "tech_support": np.random.choice([0,1], n, p=[0.6,0.4]),
    "payment_method": np.random.choice(
        ["UPI","Debit Card","Bank transfer","Credit card"],
        n
    ),
    "support_calls": np.random.randint(0, 8, n)
})

# Create total charges
data["total_charges"] = (data["monthly_charges"] * data["tenure_months"]).round(2)

# --------- REALISTIC CHURN PROBABILITY LOGIC ---------- #

# Base churn probability
churn_prob = 0.15

# Increase churn for risky behavior
risk_score = (
    (data["contract_type"] == "Month-to-month") * 0.25 +
    (data["tenure_months"] < 12) * 0.20 +
    (data["support_calls"] > 3) * 0.15 +
    (data["internet_service"] == "Fiber optic") * 0.10 +
    (data["online_security"] == 0) * 0.10 +
    (data["tech_support"] == 0) * 0.10
)

final_prob = churn_prob + risk_score

data["churn"] = np.where(
    np.random.rand(n) < final_prob,
    1,
    0
)

data.to_csv("../data/telecom_churn.csv", index=False)

print("Telecom churn dataset created with", len(data), "rows")



Telecom churn dataset created with 1500 rows


In [2]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/telecom_churn.csv")

conn = sqlite3.connect("../data/churn.db")
df.to_sql("customers", conn, if_exists="replace", index=False)
conn.close()

print("Database ready")


Database ready


## reading data from source table

In [34]:
from sklearn.preprocessing import LabelEncoder


In [1]:
import sqlite3
import pandas as pd
import requests
from datetime import datetime

def ingest_sql():
    conn = sqlite3.connect("../data/churn.db")
    df_sql = pd.read_sql("SELECT * FROM customers ", conn)

    df_sql = pd.read_sql("SELECT * FROM customers where tenure_months > 10", conn)

    conn.close()
    print("SQL rows:", len(df_sql))
    return df_sql

# def ingest_api():
#     url = "http://localhost:9100/new_customers"
#     response = requests.get(url)
#     df_api = pd.DataFrame(response.json())
#     print("API rows:", len(df_api))
#     return df_api

In [2]:
df = ingest_sql()

SQL rows: 1313


In [3]:
df.head()

,customer_id,gender,senior_citizen,tenure_months,monthly_charges,contract_type,internet_service,online_security,tech_support,payment_method,support_calls,total_charges,churn
0,1,Male,0,61,76.26,Month-to-month,DSL,0,0,Debit Card,7,4651.86,1
1,3,Male,0,68,114.04,Month-to-month,No,0,0,Debit Card,6,7754.72,1
2,4,Male,0,35,21.58,Month-to-month,Fiber optic,0,1,Debit Card,4,755.30,1
3,5,Male,0,65,72.28,Month-to-month,DSL,1,0,Bank transfer,4,4698.20,1
4,7,Male,0,45,85.61,Month-to-month,No,1,0,Debit Card,6,3852.45,1


In [4]:
df.fillna(0, inplace=True)
df.head(2)

,customer_id,gender,senior_citizen,tenure_months,monthly_charges,contract_type,internet_service,online_security,tech_support,payment_method,support_calls,total_charges,churn
0,1,Male,0,61,76.26,Month-to-month,DSL,0,0,Debit Card,7,4651.86,1
1,3,Male,0,68,114.04,Month-to-month,No,0,0,Debit Card,6,7754.72,1


In [5]:
df.contract_type.unique()

array(['Month-to-month', 'One year', 'Two year'], dtype=object)

In [6]:
from sklearn.preprocessing import LabelEncoder

# -----------------------------
# 1️⃣ Drop customer_id
# -----------------------------
df = df.drop("customer_id", axis=1)

# -----------------------------
# 2️⃣ Label Encode Gender
# -----------------------------
le = LabelEncoder()
df["gender"] = le.fit_transform(df["gender"])

# Male → 1
# Female → 0  (depending on fit order)

# -----------------------------
# 3️⃣ One-Hot Encode Categorical Columns
# -----------------------------
df = pd.get_dummies(
    df,
    columns=["contract_type", "payment_method", "internet_service"],
    dtype=int
)

df.head()


,gender,senior_citizen,tenure_months,monthly_charges,online_security,tech_support,support_calls,total_charges,churn,contract_type_Month-to-month,contract_type_One year,contract_type_Two year,payment_method_Bank transfer,payment_method_Credit card,payment_method_Debit Card,payment_method_UPI,internet_service_DSL,internet_service_Fiber optic,internet_service_No
0,1,0,61,76.26,0,0,7,4651.86,1,1,0,0,0,0,1,0,1,0,0
1,1,0,68,114.04,0,0,6,7754.72,1,1,0,0,0,0,1,0,0,0,1
2,1,0,35,21.58,0,1,4,755.30,1,1,0,0,0,0,1,0,0,1,0
3,1,0,65,72.28,1,0,4,4698.20,1,1,0,0,1,0,0,0,1,0,0
4,1,0,45,85.61,1,0,6,3852.45,1,1,0,0,0,0,1,0,0,0,1


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from xgboost import XGBClassifier
import mlflow
import mlflow.xgboost

# Split features and target
X = df.drop("churn", axis=1)
y = df["churn"]

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Enable MLflow experiment tracking
mlflow.set_experiment("simple-mlops")

with mlflow.start_run():

    # Define XGBoost model
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        use_label_encoder=False
    )

    # Hyperparameter grid
    param_grid = {
        "n_estimators": [100, 120],
        "max_depth": [3, 5],
        "learning_rate": [0.03, 0.1],
        "subsample": [0.8, 1]
    }

    # GridSearch
    grid = GridSearchCV(model, param_grid, cv=3, scoring="accuracy")
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    print("Best Parameters:", grid.best_params_)
    print("Best CV Accuracy:", grid.best_score_)
    print("Best Model:", best_model)

    # Evaluation
    preds = best_model.predict(X_test)
    print("Predictions:", preds)
    print("True Labels:", y_test.values)
    acc = accuracy_score(y_test, preds)
    recall = recall_score(y_test, preds)
    confusion = confusion_matrix(y_test, preds)

    print("Test Accuracy:", acc)
    
    # Log best parameters
    mlflow.log_params(grid.best_params_)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("tn", confusion[0][0])
    mlflow.log_metric("fp", confusion[0][1])
    mlflow.log_metric("fn", confusion[1][0])
    mlflow.log_metric("tp", confusion[1][1])


    # Log and register model
    mlflow.xgboost.log_model(
        best_model,
        artifact_path="model",
        registered_model_name="SimpleModel"
    )

    print("Training complete")
    print("Accuracy:", acc)


/Users/rakshitasingh/miniconda3/envs/mlops-env/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [01:00:43] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/rakshitasingh/miniconda3/envs/mlops-env/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [01:00:43] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/rakshitasingh/miniconda3/envs/mlops-env/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [01:00:43] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/rakshitasingh/miniconda3/envs/mlops-env/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [01:00:43] WARNING: /Users/runn

Best Parameters: {'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 120, 'subsample': 0.8}
Best CV Accuracy: 0.66
Best Model: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=120, n_jobs=None,
              num_parallel_tree=None, ...)
Predictions: [0 1 0 0 1 0 1 1 0 0 1 0 0 1 1 1 1 0 0 0 1 1 0 0 1 1 1 1 0 1 0 1 1 1 0 0 1
 0 1 

Registered model 'SimpleModel' already exists. Creating a new version of this model...
Created version '5' of model 'SimpleModel'.
